<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_P6_BOOST_STANDALONE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS P6 — Boost Standalone | Colab CPU

**Independent runner for P6 Boost r2**, reusing existing P6 W2 artifacts from `MyDrive/GEMS_2`. It does **not** rerun W0–W2, G01, G02 full research, M9 training, or W3.

**Execution:** Config → Mount & clone → Startup verification → Restore context → Stage W2 → Preflight → A → optional G02 backfill → Preflight again → B → C → D → E → Report.

First run: only `preflight` and `benchmark` enabled. No submission is uploaded by code. Stage E export defaults OFF. Source code is loaded from the selected Git ref; branch `p6-boost` must be pushed first or use a verified commit SHA on `main`.

## 0 — Controls (edit before running)

In [1]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',  # CHANGE TO 'p6-boost' after pushing r2, or preferably pin its commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False,
    'train_baselines': [],
    'run_full_inference': False,
    'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': False,
    'seed': 2026,
}
PRIVATE_REPO = False  # if True, Colab Secrets must contain GITHUB_TOKEN
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
M9_CONFIG = {'profile': 'PILOT', 'overwrite_existing': False}
P6_CONFIG = {'overwrite_existing': False}
BOOST_CONFIG = {'geology': {'backfill_missing': False}}
BOOST_RUN = {'preflight': True, 'benchmark': True, 'backfill_g02': False,
             'geology': False, 'robustness': False, 'hybrid': False,
             'selection': False, 'export': False, 'report': True}
EXPECT = {'notebook_version': 'p6-nb.3', 'code_rev': 'p6b.1-r2', 'commit': None}
# IMPORTANT: For the latest r2 workflow, CONFIG['ref'] must resolve to a commit with boost_audit.py.


## 1 — Google Drive, repository clone, version lock

Requires the exact P6 r2 implementation in the selected Git ref. A missing branch is an error. Do not silently use `main` if you selected `p6-boost`.

In [2]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
    if not token:
        raise RuntimeError('PRIVATE_REPO=True but GITHUB_TOKEN is missing in Colab Secrets')
    hdr = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True,
                       env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError(f"Git failed: {' '.join(args[:4])}\n{r.stderr[-1500:]}")
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
remote_branch = git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0
if remote_branch:
    git('-C', CODE, 'checkout', '-q', '-B', 'p6_colab_active', f'origin/{ref}')
else:
    # For a SHA/tag, require Git to resolve it; do not quietly switch to main.
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported. Restart runtime to avoid stale module code.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('Requested:', ref, '| Resolved SHA:', HEAD)


Mounted at /content/drive
Requested: main | Resolved SHA: d9445ff048fdcdb903927f5329351d56cd882341


## 2 — Startup contract

This deliberately fails with older `main` code that lacks r2. Push the latest implementation or select its exact commit before proceeding.

In [3]:
import json
from IPython.display import display
import pandas as pd
from gems_bench.p6.boost_audit import startup_check
START = startup_check(CONFIG['ref'], HEAD,
    {**EXPECT, 'code_root': CODE,
     'artifact_root': f"{CONFIG['drive_root']}/outputs/p6_fast_track/boost"}, BOOST_CONFIG)
print(json.dumps(START, indent=2, default=str))


{
  "requested_ref": "main",
  "resolved_commit": "d9445ff048fdcdb903927f5329351d56cd882341",
  "installed_commit": "d9445ff048fdcdb903927f5329351d56cd882341",
  "boost_module": "/content/GEMS/gems_bench/p6/boost.py",
  "boost_code_rev": "p6b.1-r2",
  "boost_version": "p6b.1",
  "boost_config_fp": "2f973e4709ae",
  "notebook_version": "p6-nb.3",
  "artifact_root": "/content/drive/MyDrive/GEMS_2/outputs/p6_fast_track/boost",
  "problems": []
}


## 3 — Reattach existing M9/P6 context (no retraining)

In [4]:
from gems_bench.pipeline import Pipeline
from gems_bench.p6 import P6FastTrack

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
for step in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(step)  # validate + stage cached Drive data, never rebuild
m9 = pipe.m9(M9_CONFIG)
m9.load()
p6 = P6FastTrack(m9, P6_CONFIG)
print('P6 workspace:', p6.ws.root)
display(p6.budget.table())


[sync out] 2 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/config_snapshot.json', 'reports/pipeline/environment_report.csv']
[sync out] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/stages/bootstrap.json']
[bootstrap] DONE: bootstrap: ok (3 checks)
[bootstrap] outputs: ['/content/drive/MyDrive/GEMS_2/environment/pipeline/config_snapshot.json', '/content/drive/MyDrive/GEMS_2/reports/pipeline/environment_report.csv']
[sync out] 2 copied (0.0 MiB), 0 unchanged, 0s: ['data/raw/acquisition_manifest.json', 'reports/pipeline/raw_data_acquisition.csv']
[sync out] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/stages/raw_data.json']
[raw_data] DONE: raw_data: ok (3 checks)
[raw_data] outputs: ['/content/drive/MyDrive/GEMS_2/data/raw/acquisition_manifest.json', '/content/drive/MyDrive/GEMS_2/reports/pipeline/raw_data_acquisition.csv']
[sync in] 1 copied (0.0 MiB), 0 unchanged, 0s: ['environment/pipeline/native_grid.json']
[native_grid] SKIP_CACHED: native_g

,workstream,device,cap_h,spent_h,left_h
0,w0,gpu,0.0,0.0,0.0
1,w0,cpu,0.5,0.0,0.5
2,w1,gpu,4.0,0.0,4.0
3,w1,cpu,1.5,0.0,1.5
4,w2,gpu,0.0,0.0,0.0
5,w2,cpu,6.0,0.0,6.0
6,w3,gpu,2.0,0.0,2.0
7,w3,cpu,0.5,0.0,0.5
8,w4,gpu,0.0,0.0,0.0
9,w4,cpu,6.0,0.0,6.0


## 4 — Restore frozen W2 inputs

A–E depend on the *existing* W2 candidate JSON, sampled rows and 6 stored prediction maps. Missing inputs stop here: do **not** regenerate W2 in this notebook.

In [5]:
bst = p6.boost(BOOST_CONFIG)
need = bst.stage()
required = ['stack/candidate.json', 'stack/features/rows.parquet']
missing = [name for name in required if not need.get(name)]
if need.get('stack/maps/*.npy', 0) != 6:
    missing.append(f"stack/maps/*.npy expected 6, got {need.get('stack/maps/*.npy', 0)}")
if missing:
    raise RuntimeError('Frozen W2 artifacts missing: ' + ', '.join(missing) +
                       '. Restore P6 W2 artifacts on Drive; do not rerun W2.')
print('Restored:', need)
print('Champion:', json.dumps(bst.champion(), indent=2, default=str))


[sync in] 45 copied (340.0 MiB), 0 unchanged, 25s: ['outputs/p6_fast_track/stack', 'outputs/p6_fast_track/probes', 'outputs/p6_fast_track/submissions', 'outputs/p6_fast_track/manifests', 'outputs/p6_fast_track/selection']
[boost] W2 inputs: {'stack/candidate.json': True, 'stack/features/rows.parquet': True, 'stack/maps/*.npy': 6}
Restored: {'stack/candidate.json': True, 'stack/features/rows.parquet': True, 'stack/maps/*.npy': 6}
Champion: {
  "disc_dti": 0.224640288469153,
  "disc_worst": 0.15341439472074078,
  "geo_dti": 0.18084266130488846,
  "source": "stack/candidate.json"
}


## 5 — Preflight (run before A; rerun after any G02 backfill)

Expected known issues: G02 zone-2 PARTIAL; W2 submitted representation was skeleton@5%, not its nested discovery choice. Investigate BLOCKED evaluation contracts before selection.

In [6]:
if BOOST_RUN['preflight']:
    pre = bst.preflight()
    print(json.dumps(pre['areas'], indent=2, default=str))
    display(bst.load_table('feature_compatibility', 'preflight'))
    sp = (bst.read_json('preflight', 'submission_provenance') or {}).get('w2', {})
    print('W2 submission provenance:', json.dumps({k: sp.get(k) for k in
          ('status', 'rep', 'frac', 'blend', 'selected_cell', 'interpretation')}, indent=2, default=str))
    print('G02 backfill plan:', json.dumps(bst.read_json('preflight','g02_backfill_plan'), indent=2, default=str)[:4000])


[sync in] 333 copied (4168.4 MiB), 0 unchanged, 219s: ['cache/m9/layers', 'outputs/m9_geofault_production/research/G01_wavelet_geology', 'outputs/m9_geofault_production/research/G02_structural_geology']
[g02] attached g02_23686a63cb (explicit run_id); unit rasters: ['disc_official_0', 'disc_official_1', 'geo_0', 'geo_1']


/content/GEMS/gems_bench/p6/probes.py:284: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  t = pd.concat([x.reindex(columns=LOG_COLUMNS) for x in parts], ignore_index=True)


[sync out] 58 copied (340.1 MiB), 0 unchanged, 4s: ['outputs/p6_fast_track']
{
  "g02_attachment": "BLOCKED",
  "g02_backfill_plan": "BLOCKED",
  "dem_coverage": "WARNING",
  "feature_contract": "BLOCKED",
  "raster_alignment": "BLOCKED",
  "train_inference_parity": "WARNING",
  "evaluation_contract": "PASS",
  "submission_provenance": "WARNING"
}


,candidate,features,status,missing_training,unverified,inference_absent,inference_to_compute,note
0,S0,cnn_b0;cnn_b1;cnn_disagree;mag_hgrad;grav_hgra...,PASS,NaN,NaN,NaN,NaN,NaN
1,S1,cnn_b0;cnn_b1;cnn_disagree;mag_hgrad;grav_hgra...,BLOCKED,geo:2:g02c;disc_official:2:g02c,NaN,NaN,g02c,uses G02 rasters
2,S2,cnn_b0;cnn_b1;cnn_disagree;mag_hgrad;grav_hgra...,BLOCKED,geo:2:g02c;disc_official:2:g02c;geo:2:g02a;dis...,NaN,NaN,g02c;g02a,uses G02 rasters
3,S3,cnn_b0;cnn_b1;cnn_disagree;mag_hgrad;grav_hgra...,BLOCKED,geo:2:g02c;disc_official:2:g02c;geo:2:g02a;dis...,NaN,NaN,g02c;g02a;g01_persistence,uses G02 rasters
4,C1,cnn_b0;cnn_b1;cnn_disagree;mag_hgrad;grav_hgra...,PASS,NaN,NaN,NaN,NaN,NaN
5,C2,cnn_b0;cnn_b1;cnn_disagree;mag_hgrad;grav_hgra...,PASS,NaN,NaN,NaN,NaN,NaN
6,D1/D2 (on the stage-C base),base features + optional g02c layer,PASS,NaN,NaN,NaN,NaN,D2 drops its geology term when g02c is absent ...
7,dem_* (no candidate),NaN,NOT_IN_ANY_CANDIDATE,NaN,NaN,NaN,NaN,optional adapter only; a DEM variant needs its...


W2 submission provenance: {
  "status": "VERIFIED",
  "rep": "skeleton_1px",
  "frac": 0.05,
  "blend": 1.0,
  "selected_cell": false,
  "interpretation": "the recorded W2 numbers are nested choices; the uploaded file is the cell skeleton_1px @ 0.05 (blend 1.0), which NO nested evaluation selected: its offline score is the matched cell above"
}
G02 backfill plan: {
  "switch": "BOOST_CONFIG['geology']['backfill_missing']",
  "enabled": false,
  "max_units_per_call": 2,
  "executes_in_preflight": false,
  "label_independent": true,
  "rewrites_existing_rasters": false,
  "resume": "per-tile MapStore under processed/tmp_<unit>; finished rasters are skipped",
  "status": "BLOCKED",
  "units": [
    {
      "unit": "geo:2",
      "output": "/content/gems_work/outputs/m9_geofault_production/research/G02_structural_geology/g02_23686a63cb/rasters/units/geo_2.npz",
      "refs": "processed/refs_geo_2.json",
      "done_marker": "processed/done_raster_geo_2",
      "px": 6351156,
      "est_out

## 6 — A: Matched ranking × representation × budget

Read `boost/benchmark/verdict.json` first. Decide whether observed W2 advantage comes from ranking or representation/budget. No new DTI is claimed until this cell runs on the real data.

In [7]:
if BOOST_RUN['benchmark']:
    a = bst.a_benchmark()
    display(a['ranking_gain'])
    display(a['nested'][['space','disc_dti','disc_worst','geo_dti','systems_gained','systems_lost']])
    print('VERDICT:', json.dumps(a['verdict'], indent=2, default=str))


[sync in] 1 copied (0.0 MiB), 0 unchanged, 0s: ['outputs/m9_geofault_production/baseline/baseline_selection.json']
[sync out] 15 copied (0.6 MiB), 58 unchanged, 0s: ['outputs/p6_fast_track']
[sync out] 2 copied (0.0 MiB), 72 unchanged, 0s: ['outputs/p6_fast_track']


,task,component,from,to,from_dti,to_dti,gain,worst_gain,zones_improved,n_zones,zone_gains
0,disc_official,published_to_matched_grid,fallback_published,fallback_fusion|crest_1px,0.161576,0.162482,0.000906,0.000000,2.0,3.0,"{""0"": 0.001766, ""1"": 0.005527, ""2"": 0.0}"
1,disc_official,representation,fallback_fusion|crest_1px,fallback_fusion|rep+frac,0.162482,0.162482,0.000000,0.000000,0.0,3.0,"{""0"": 0.0, ""1"": 0.0, ""2"": 0.0}"
2,disc_official,ranking_at_crest,fallback_fusion|crest_1px,w2|crest_1px,0.162482,0.238882,0.076400,0.066415,3.0,3.0,"{""0"": 0.083923, ""1"": 0.078806, ""2"": 0.066415}"
3,disc_official,ranking_at_skeleton,fallback_fusion|skeleton_1px,w2|skeleton_1px,0.127711,0.242156,0.114445,0.094170,3.0,3.0,"{""0"": 0.128337, ""1"": 0.122889, ""2"": 0.09417}"
4,disc_official,ranking,fallback_fusion|rep+frac,w2|rep+frac,0.162482,0.242156,0.079674,0.072420,3.0,3.0,"{""0"": 0.08852, ""1"": 0.08064, ""2"": 0.07242}"
5,disc_official,hybridization,w2|rep+frac,w2_space|blend+rep+frac,0.242156,0.224640,-0.017516,-0.045165,0.0,3.0,"{""0"": 0.0, ""1"": 0.0, ""2"": -0.045165}"
6,disc_official,total,fallback_published,w2_space|blend+rep+frac,0.161576,0.224640,0.063064,0.027255,3.0,3.0,"{""0"": 0.090286, ""1"": 0.086167, ""2"": 0.027255}"
7,disc_official,matched_cells_w2_minus_fusion,fallback_fusion,w2,NaN,NaN,0.091027,0.008763,10.0,10.0,"{""crest_1px@0.005"": 0.132031, ""crest_1px@0.01""..."
8,disc_official,cell:w2_disc_choice,fallback_fusion,w2,0.074009,0.242156,0.168147,NaN,NaN,NaN,"{""rep"": ""skeleton_1px"", ""frac"": 0.01}"
9,disc_official,cell:w2_geo_choice,fallback_fusion,w2,0.133039,0.141802,0.008763,NaN,NaN,NaN,"{""rep"": ""crest_1px"", ""frac"": 0.05}"


,space,disc_dti,disc_worst,geo_dti,systems_gained,systems_lost
0,fallback_fusion|crest_1px,0.162482,0.126160,0.182832,0,4
1,fallback_fusion|skeleton_1px,0.127711,0.104409,0.169631,16,0
2,fallback_fusion|rep+frac,0.162482,0.126160,0.182832,0,4
3,cnn_b0|crest_1px,0.150901,0.094924,0.180474,3,5
4,cnn_b0|skeleton_1px,0.111917,0.064103,0.163262,4,5
5,cnn_b0|rep+frac,0.150901,0.094924,0.180474,3,5
6,cnn_b1|crest_1px,0.144669,0.116399,0.183458,2,7
7,cnn_b1|skeleton_1px,0.108214,0.088186,0.167385,3,2
8,cnn_b1|rep+frac,0.144669,0.116399,0.183458,2,7
9,w2|crest_1px,0.238882,0.192575,0.185609,5,3


VERDICT: {
  "task": "disc_official",
  "ranking_gain": 0.07967374202648361,
  "zones_improved": 3,
  "cell_win_share": 1.0,
  "largest_component": "ranking",
  "components": {
    "published_to_matched_grid": 0.0009060885869359159,
    "representation": 0.0,
    "ranking": 0.07967374202648361,
    "hybridization": -0.017515754827356983
  },
  "w2_gain_is_ranking": true,
  "rule": {
    "min_ranking_gain": 0.005,
    "min_zones_improved": 2,
    "min_cell_win_share": 0.6
  },
  "note": "nested numbers use the matched fracs only; W2's recorded 0.224640 also searched fracs 0.02 / 0.075, so w2_space here may differ slightly"
}


## 7 — B0: Optional G02 zone-2 backfill

**OFF by default.** Inspect `preflight/g02_backfill_plan.json`, validate original-input compatibility and remaining budget first. Backfill is an explicit heavy computation; run preflight again afterward.

In [8]:
if BOOST_RUN['backfill_g02']:
    plan = bst.read_json('preflight','g02_backfill_plan') or {}
    assert plan.get('status') in ('PENDING','READY'), (
        f"Backfill not permitted: {plan.get('status')} — {plan.get('reason') or plan.get('input_drift')}")
    BOOST_CONFIG = {**BOOST_CONFIG, 'geology': {**BOOST_CONFIG.get('geology', {}),
                                               'backfill_missing': True}}
    bst = p6.boost(BOOST_CONFIG)
    completed = bst.section('geology_backfill', bst.backfill)
    print('Backfill result:', completed)
    print('Re-run cell 5 (preflight) before B.')


## 8 — B: G02-C / G02-A / G01 geological ablation

S0–S3 reuse W2 sampled rows. If any required zone raster is absent, the affected variant must be BLOCKED, not evaluated on fewer zones.

In [9]:
if BOOST_RUN['geology']:
    b = bst.b_geology()
    print(json.dumps({k: v for k, v in b['attach'].items() if k != 'root_cause'}, indent=2, default=str))
    display(b['audit'].pivot_table(index='unit', columns='feature', values='key_present', aggfunc='first'))
    display(b['ablation'][['candidate','status','decision','disc_dti','disc_worst',
                           'geo_dti','gain_vs_base','reasons']])
    print('Choice:', b['decision']['chosen'], '| Status:', bst.read_json('geology', 'status'))


## 9 — C: Zone robustness diagnostics and ablations

No zone-specific threshold tuning. Inspect weak-zone metric, input shifts and geographic preservation.

In [10]:
if BOOST_RUN['robustness']:
    c = bst.c_robustness()
    for name in ('zone_ranking','composition','prediction_geometry'):
        display(c['diagnostics'][name])
    display(c['ablation'][['candidate','decision','disc_dti','disc_worst','geo_dti','reasons']])


## 10 — D: Fault-system hybrid recovery

Compare recovered and lost systems under the *same eligible prediction area*. Reject unacceptable false-positive increases.

In [11]:
if BOOST_RUN['hybrid']:
    d = bst.d_hybrid()
    display(d['ablation'][['candidate','decision','disc_dti','disc_worst','geo_dti',
                           'systems_gained','systems_lost','fp_increase','reasons']])
    display(d['d3'])


## 11 — E: Candidate selection (export OFF by default)

Pass all provenance, parity, nested DTI, worst-zone and geographic gates before enabling export. Never upload automatically.

In [12]:
if BOOST_RUN['selection']:
    e = bst.e_select(export=BOOST_RUN['export'])
    print(json.dumps({k: v for k, v in e.items() if k != 'champion'}, indent=2, default=str)[:5000])
    display(bst.load_table('candidates','selection'))
    display(bst.load_table('leaderboard_compare','selection'))
else:
    assert not BOOST_RUN['export'], 'Set selection=True before export=True'


## 12 — Reports, indexes, and saved artifacts

All outputs persist to Drive under `outputs/p6_fast_track/boost/`. Reports must never invent measurements for stages not yet run.

In [13]:
if BOOST_RUN['report']:
    rep = bst.report()
    display(rep['status'])
    display(rep['summary'])
    print('Readiness:', json.dumps(rep['readiness'].get('questions', {}), indent=2, default=str)[:5000])
    print('Artifact directory:', p6.ws.root / 'boost')


[sync out] 10 copied (0.4 MiB), 74 unchanged, 0s: ['outputs/p6_fast_track']


,stage,status,fingerprint,reason,updated,seconds,source_commit
0,preflight,COMPLETE,,,2026-10-27T20:13:38,231.3,d9445ff048fdcdb903927f5329351d56cd882341
1,benchmark,COMPLETE,b67365220cbf,,2026-10-27T20:19:30,348.0,d9445ff048fdcdb903927f5329351d56cd882341
2,geology,PENDING,,,,NaN,None
3,robustness,PENDING,,,,NaN,None
4,hybrid,PENDING,,,,NaN,None
5,selection,PENDING,,,,NaN,None


,stage,candidate,disc_dti,geo_dti,decision,disc_worst,systems_gained,systems_lost
0,recorded,P5 B0+B1 fallback,0.161576,0.182832,fallback,NaN,NaN,NaN
1,recorded,P6 W2 LightGBM,0.224640,0.180843,champion,0.153414,NaN,NaN
2,benchmark,fallback_fusion|crest_1px,0.162482,0.182832,diagnostic,0.126160,0.0,4.0
3,benchmark,fallback_fusion|skeleton_1px,0.127711,0.169631,diagnostic,0.104409,16.0,0.0
4,benchmark,fallback_fusion|rep+frac,0.162482,0.182832,diagnostic,0.126160,0.0,4.0
5,benchmark,cnn_b0|crest_1px,0.150901,0.180474,diagnostic,0.094924,3.0,5.0
6,benchmark,cnn_b0|skeleton_1px,0.111917,0.163262,diagnostic,0.064103,4.0,5.0
7,benchmark,cnn_b0|rep+frac,0.150901,0.180474,diagnostic,0.094924,3.0,5.0
8,benchmark,cnn_b1|crest_1px,0.144669,0.183458,diagnostic,0.116399,2.0,7.0
9,benchmark,cnn_b1|skeleton_1px,0.108214,0.167385,diagnostic,0.088186,3.0,2.0


Readiness: {
  "q1_w2_better_than_fallback_matched": {
    "status": "ANSWERED",
    "answer": true,
    "ranking_gain": 0.07967374202648361,
    "zones_improved": 3,
    "cell_win_share": 1.0,
    "evidence": "benchmark/verdict.json"
  },
  "q2_ranking_vs_representation_budget": {
    "status": "ANSWERED",
    "components": {
      "published_to_matched_grid": 0.0009060885869359159,
      "representation": 0.0,
      "ranking": 0.07967374202648361,
      "hybridization": -0.017515754827356983
    },
    "largest": "ranking",
    "evidence": "benchmark/ranking_gain.csv"
  },
  "q3_g02c_improves_discovery": {
    "status": "PENDING",
    "answer": null
  },
  "q4_g02a_incremental": {
    "status": "PENDING",
    "answer": null
  },
  "q5_g01_persistence_independent": {
    "status": "PENDING",
    "answer": null
  },
  "q6_zone2_improving": {
    "status": "PENDING",
    "rows": [],
    "w2_zone2_recorded": 0.153414
  },
  "q7_additional_systems": {
    "status": "ANSWERED",
    "rows":

## Execution decision checklist

1. First run: `preflight=True`, `benchmark=True`; all other compute stages OFF. Review preflight + A verdict.
2. Only if justified: enable `backfill_g02=True` (after inspecting plan), rerun preflight, enable geology.
3. When B has ACCEPT/REJECT: enable C and D individually; keep export OFF.
4. Enable E with export OFF; if all gates pass, rerun E with `export=True`. Upload submission **manually**.

**Baselines recorded (not measured by this notebook):** fallback discovery 0.161576, geographic 0.182832; W2 discovery 0.224640, worst zone 0.153414, geographic 0.180843.

**Source:** `docs/p6/P6_BOOST_HANDOFF.md`. This notebook is a new runner; repository r2 code must be available at selected Git ref.
